# Notebook 02: Análise de Dados de Saúde — DataSUS

**Projeto:** `migracao-venezuelana-oeste-sc`  
**Autores:** Leonardo Rafael Santos Leitão e Vicente Neves da Silva Ribeiro (UFFS)  
**Data:** Abril de 2026

---

## Objetivo

Este notebook explora dados do DataSUS para o Oeste de Santa Catarina, focando nos sistemas:

- **SIM** (Sistema de Informação sobre Mortalidade): óbitos;
- **SINASC** (Sistema de Informação sobre Nascidos Vivos): nascimentos;
- **SIH/AIH** (Sistema de Informações Hospitalares): internações hospitalares.

As análises cobrem o período **2018–2024** e incluem séries temporais, indicadores de saúde e comparações entre população venezuelana e brasileira.

> **Nota metodológica:** Os dados apresentados são **placeholder/simulados**. Os dados reais serão obtidos via `pysus` e FTP do DataSUS.

## 1. Configuração do Ambiente

In [ ]:
import os
import sys
from pathlib import Path
from typing import Optional, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 100

PROJECT_ROOT = Path(".").resolve().parent
DATA_DIR = PROJECT_ROOT / "data"
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

ANOS = list(range(2018, 2025))
MESES = list(range(1, 13))

print(f"Período de análise: {ANOS[0]}–{ANOS[-1]}")
print(f"Figuras serão salvas em: {FIGURES_DIR}")

## 2. Dados Sintéticos — Placeholder

Simulamos séries temporais mensais de óbitos, nascimentos e internações para os municípios do Oeste de SC.

In [ ]:
np.random.seed(42)

municipios = ["Chapecó", "Xanxerê", "Concórdia", "Joaçaba", "São Miguel do Oeste"]

records = []
for ano in ANOS:
    for mes in MESES:
        for mun in municipios:
            # Tendência crescente até 2022, estabilização depois
            base = 1.0 + 0.15 * (ano - 2018) if ano <= 2022 else 1.6
            seasonal = 1.0 + 0.2 * np.sin(2 * np.pi * mes / 12)
            
            records.append({
                "ano": ano,
                "mes": mes,
                "data": pd.Timestamp(year=ano, month=mes, day=15),
                "municipio": mun,
                "nacionalidade": "Venezuelana",
                "obitos": max(0, int(np.random.poisson(3 * base * seasonal))),
                "nascimentos": max(0, int(np.random.poisson(5 * base * seasonal))),
                "internacoes": max(0, int(np.random.poisson(12 * base * seasonal))),
                "dias_permanencia": max(1, int(np.random.gamma(3, 2))),
                "valor_aih": round(np.random.gamma(5, 500), 2),
            })
            records.append({
                "ano": ano,
                "mes": mes,
                "data": pd.Timestamp(year=ano, month=mes, day=15),
                "municipio": mun,
                "nacionalidade": "Brasileira",
                "obitos": max(0, int(np.random.poisson(45 * seasonal))),
                "nascimentos": max(0, int(np.random.poisson(80 * seasonal))),
                "internacoes": max(0, int(np.random.poisson(200 * seasonal))),
                "dias_permanencia": max(1, int(np.random.gamma(4, 2.5))),
                "valor_aih": round(np.random.gamma(6, 550), 2),
            })

df = pd.DataFrame(records)
df["ano_mes"] = df["data"].dt.to_period("M")

print(f"Dimensões: {df.shape}")
display(df.head())
print("\nResumo por nacionalidade:")
display(df.groupby("nacionalidade")[["obitos", "nascimentos", "internacoes"]].sum())

## 3. Série Temporal de Óbitos (SIM)

In [ ]:
# -----------------------------------------------------------------------------
# Série temporal: óbitos por nacionalidade
# -----------------------------------------------------------------------------
obitos_ts = df.groupby(["data", "nacionalidade"])["obitos"].sum().reset_index()
obitos_pivot = obitos_ts.pivot(index="data", columns="nacionalidade", values="obitos").fillna(0)

fig, ax = plt.subplots(figsize=(14, 6))
obitos_pivot["Venezuelana"].plot(ax=ax, label="Venezuelana", color="#d62728", linewidth=2, marker="o", markersize=4)
obitos_pivot["Brasileira"].plot(ax=ax, label="Brasileira", color="#1f77b4", linewidth=2, alpha=0.7)

ax.set_title("Óbitos por Nacionalidade — Oeste de SC (Placeholder)", fontsize=14, fontweight="bold")
ax.set_xlabel("Ano")
ax.set_ylabel("Número de óbitos mensais")
ax.legend(title="Nacionalidade")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
sns.despine()
plt.tight_layout()

fig_path = FIGURES_DIR / "02_obitos_serie_temporal.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="white")
print(f"Figura salva: {fig_path}")
plt.show()

## 4. Série Temporal de Nascimentos (SINASC)

In [ ]:
# -----------------------------------------------------------------------------
# Série temporal: nascimentos por nacionalidade
# -----------------------------------------------------------------------------
nasc_ts = df.groupby(["data", "nacionalidade"])["nascimentos"].sum().reset_index()
nasc_pivot = nasc_ts.pivot(index="data", columns="nacionalidade", values="nascimentos").fillna(0)

fig, ax = plt.subplots(figsize=(14, 6))
nasc_pivot["Venezuelana"].plot(ax=ax, label="Venezuelana", color="#2ca02c", linewidth=2, marker="s", markersize=4)
nasc_pivot["Brasileira"].plot(ax=ax, label="Brasileira", color="#1f77b4", linewidth=2, alpha=0.7)

ax.set_title("Nascimentos por Nacionalidade — Oeste de SC (Placeholder)", fontsize=14, fontweight="bold")
ax.set_xlabel("Ano")
ax.set_ylabel("Número de nascimentos mensais")
ax.legend(title="Nacionalidade")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
sns.despine()
plt.tight_layout()

fig_path = FIGURES_DIR / "02_nascimentos_serie_temporal.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="white")
print(f"Figura salva: {fig_path}")
plt.show()

## 5. Hospitalizações (AIH) — Volume e Valor

In [ ]:
# -----------------------------------------------------------------------------
# Internações: volume e valor médio da AIH
# -----------------------------------------------------------------------------
aih_ts = df.groupby(["data", "nacionalidade"]).agg({
    "internacoes": "sum",
    "valor_aih": "mean",
    "dias_permanencia": "mean",
}).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (col, title, ylabel) in enumerate([
    ("internacoes", "Internações mensais", "Quantidade"),
    ("valor_aih", "Valor médio da AIH (R$)", "R$"),
    ("dias_permanencia", "Dias médios de permanência", "Dias"),
]):
    pivot = aih_ts.pivot(index="data", columns="nacionalidade", values=col).fillna(0)
    pivot["Venezuelana"].plot(ax=axes[i], color="#ff7f0e", linewidth=2, marker="D", markersize=3, label="Venezuelana")
    pivot["Brasileira"].plot(ax=axes[i], color="#1f77b4", linewidth=2, alpha=0.6, label="Brasileira")
    axes[i].set_title(title, fontweight="bold")
    axes[i].set_xlabel("Ano")
    axes[i].set_ylabel(ylabel)
    axes[i].legend()
    axes[i].xaxis.set_major_locator(mdates.YearLocator())
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    sns.despine(ax=axes[i])

fig.suptitle("Indicadores Hospitalares — Oeste de SC (Placeholder)", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()

fig_path = FIGURES_DIR / "02_hospitalizacoes_indicadores.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="white")
print(f"Figura salva: {fig_path}")
plt.show()

## 6. Agregação Anual e Comparativo por Município

In [ ]:
# -----------------------------------------------------------------------------
# Agregação anual
# -----------------------------------------------------------------------------
df_anual = df.groupby(["ano", "municipio", "nacionalidade"]).agg({
    "obitos": "sum",
    "nascimentos": "sum",
    "internacoes": "sum",
}).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (col, title) in enumerate([
    ("obitos", "Óbitos anuais por município"),
    ("nascimentos", "Nascimentos anuais por município"),
    ("internacoes", "Internações anuais por município"),
]):
    sub = df_anual[df_anual["nacionalidade"] == "Venezuelana"]
    sns.barplot(data=sub, x="ano", y=col, hue="municipio", ax=axes[i], palette="tab10")
    axes[i].set_title(title, fontweight="bold")
    axes[i].set_xlabel("Ano")
    axes[i].set_ylabel("Quantidade")
    axes[i].legend(title="Município", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
    sns.despine(ax=axes[i])

fig.suptitle("Eventos de Saúde — População Venezuelana por Município (Placeholder)", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()

fig_path = FIGURES_DIR / "02_saude_por_municipio.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="white")
print(f"Figura salva: {fig_path}")
plt.show()

## 7. Salvamento dos Dados Agregados

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

out_path = DATA_DIR / "gold" / "datasus_series_temporais.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)
print(f"Dados exportados: {out_path}")

out_anual = DATA_DIR / "gold" / "datasus_anual_municipio.csv"
df_anual.to_csv(out_anual, index=False)
print(f"Dados anuais exportados: {out_anual}")

print("\n--- Execução concluída ---")

## Próximos Passos

- [ ] Instalar e configurar `pysus` para download automatizado dos arquivos DBC;
- [ ] Descompactar e converter arquivos SIM, SINASC e SIH para Parquet;
- [ ] Cruzar dados de nacionalidade (quando disponíveis) ou inferir via algoritmos de correspondência;
- [ ] Calcular taxas específicas de mortalidade e fecundidade por idade;
- [ ] Analisar causas de óbito (CID-10) e procedimentos hospitalares (SIGTAP);
- [ ] Validar com dados do SIM/SINASC estaduais de SC.